# 06 Final Core Hybrid Metrics - Two Models

Run all cells once.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# =========================
# 1. Config
# =========================

K_VALUES = [10, 20]

INPUT_FILE = "../data_outputs/13_candidate_job_hybrid_ranking.xlsx"

OUTPUT_WITH_LABELS = "../data_outputs/13_candidate_job_final_ranking_with_labels_simple.xlsx"
OUTPUT_SUMMARY = "../data_outputs/16_final_metrics_summary_simple.xlsx"

DIRECT_MATCH_THRESHOLD = 0.50
TAXONOMY_GROUP_THRESHOLD = 0.50


# =========================
# 2. Metric functions
# =========================

def precision_at_k(labels, k):
    """
    Trong top K job, có bao nhiêu job relevant.
    """
    top_k = labels[:k]
    return top_k.sum() / k


def recall_at_k(labels, total_relevant, k):
    """
    Trong toàn bộ job relevant, top K lấy được bao nhiêu.
    """
    if total_relevant == 0:
        return 0.0
    top_k = labels[:k]
    return top_k.sum() / total_relevant


def ndcg_at_k(labels, total_relevant, k):
    """
    Đánh giá job relevant có được xếp cao không.
    """
    top_k = labels[:k]

    if len(top_k) == 0 or total_relevant == 0:
        return 0.0

    # DCG: điểm ranking thực tế
    discounts = np.log2(np.arange(2, len(top_k) + 2))
    dcg = np.sum(top_k / discounts)

    # IDCG: điểm ranking lý tưởng
    ideal_len = min(total_relevant, k)
    ideal_labels = np.ones(ideal_len)
    ideal_discounts = np.log2(np.arange(2, ideal_len + 2))
    idcg = np.sum(ideal_labels / ideal_discounts)

    return dcg / idcg if idcg > 0 else 0.0


def auc_score(labels, scores):
    """
    Đo khả năng phân biệt relevant và non-relevant.
    """
    if len(np.unique(labels)) < 2:
        return np.nan
    return roc_auc_score(labels, scores)


# =========================
# 3. Read ranking file from file 04
# =========================

df = pd.read_excel(INPUT_FILE)

print("Input shape:", df.shape)
print("Columns:", list(df.columns))


# =========================
# 4. Create relevant label
# =========================

df["direct_match"] = df["skill_overlap_score"] >= DIRECT_MATCH_THRESHOLD # Nếu candidate trùng skill với job đủ nhiều thì coi là match trực tiếp.

# Ví dụ job cần 4 skill, candidate có 2 skill trùng: skill_overlap_score = 2/4 = 0.5 → direct_match = True
# → direct_match = True

# Nếu skill overlap chưa đủ cao, nhưng candidate và job vẫn cùng nhóm taxonomy mạnh thì vẫn coi là phù hợp.
df["taxonomy_match"] = (
    (df["skill_overlap_score"] < DIRECT_MATCH_THRESHOLD) # Skill chưa trùng nhiều.
    & (df["group_similarity_score"] >= TAXONOMY_GROUP_THRESHOLD) # Nhóm taxonomy giống nhau đủ nhiều.
    & (df["dominant_group_score"] == 1) # Nhóm chuyên môn chính giống nhau là tín hiệu bổ sung mạnh.
)

# Ví dụ:

#Candidate: React, Vue
#Job: Angular, TypeScript

#Skill có thể không trùng nhiều, nhưng đều thuộc nhóm:

#Software Development / Frontend

#nên có thể được coi là taxonomy_match.


df["relevant"] = (df["direct_match"] | df["taxonomy_match"]).astype(int) 
# Nếu direct_match đúng
# HOẶC taxonomy_match đúng
# → relevant = 1

print("Relevant pairs:", df["relevant"].sum())


# =========================
# 5. Define two models
# =========================

models = [
    {
        "model": "Model 1 - Skill-only Baseline",
        "score_col": "skill_only_score",
        "description": "Only direct skill overlap; no taxonomy; no semantic embedding.",
    },
    {
        "model": "Model 2 - Core Hybrid Recommendation Model",
        "score_col": "offline_final_score",
        "description": "Skill + Taxonomy + Semantic Embedding.",
    },
]


# =========================
# 6. Evaluate each model
# =========================

summary_rows = []

for model_info in models:
    model_name = model_info["model"]
    score_col = model_info["score_col"]

    candidate_rows = []

    for candidate_id, group in df.groupby("candidate_id"):
        # Sort jobs by model score
        ranked = group.sort_values(score_col, ascending=False).copy()

        labels = ranked["relevant"].to_numpy()
        scores = ranked[score_col].to_numpy()
        total_relevant = labels.sum()

        row = {
            "candidate_id": candidate_id,
            "model": model_name,
            "AUC": auc_score(labels, scores),
        }

        for k in K_VALUES:
            row[f"Precision@{k}"] = precision_at_k(labels, k)
            row[f"Recall@{k}"] = recall_at_k(labels, total_relevant, k)
            row[f"NDCG@{k}"] = ndcg_at_k(labels, total_relevant, k)

        candidate_rows.append(row)

    candidate_metrics = pd.DataFrame(candidate_rows)

    # Average metrics across candidates
    summary = {
        "model": model_name,
        "score_column": score_col,
        "description": model_info["description"],
        "num_candidates": df["candidate_id"].nunique(),
        "num_pairs": len(df),
        "num_relevant_pairs": int(df["relevant"].sum()),
    }

    metric_cols = [
        "AUC",
        "Precision@10", "Recall@10", "NDCG@10",
        "Precision@20", "Recall@20", "NDCG@20",
    ]

    for col in metric_cols:
        summary[col] = candidate_metrics[col].mean()

    summary_rows.append(summary)


summary_df = pd.DataFrame(summary_rows)


# =========================
# 7. Save outputs
# =========================

df.to_excel(OUTPUT_WITH_LABELS, index=False)
summary_df.to_excel(OUTPUT_SUMMARY, index=False)

print("\nSummary:")
print(summary_df.to_string(index=False))

Input shape: (400, 28)
Columns: ['candidate_id', 'job_id', 'skill_overlap_score', 'group_similarity_score', 'dominant_group_score', 'skill_only_score', 'taxonomy_score', 'semantic_similarity', 'pipeline_final_score', 'semantic_score_norm', 'core_recommendation_score', 'offline_final_score', 'final_score', 'rank_skill_only', 'rank_core_hybrid', 'final_rank', 'recommendation_rank', 'job_title', 'candidate_mapped_skills', 'job_mapped_skills', 'candidate_groups', 'job_groups', 'candidate_dominant_group', 'job_dominant_group', 'match_explanation', 'semantic_rank', 'hybrid_score', 'hybrid_rank']
Relevant pairs: 236

Summary:
                                     model        score_column                                                    description  num_candidates  num_pairs  num_relevant_pairs      AUC  Precision@10  Recall@10  NDCG@10  Precision@20  Recall@20  NDCG@20
             Model 1 - Skill-only Baseline    skill_only_score Only direct skill overlap; no taxonomy; no semantic embeddin